In [1]:
import sys 
import os 

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd 
import numpy as np
import unicodedata
import re
from pathlib import Path

from config.data_cleaning_config import (
    DEFAULT_DROP_COLUMNS,
    COLUNAS_TEXTO,
    COLUNAS_BINARIAS,
    COLUNAS_INT,
    COLUNA_HORA,
    MAPA_DIA_SEMANA
)


In [3]:
data_dir = Path("../data/raw")
arquivos = list(data_dir.glob("*.csv"))

In [4]:
output_dir = Path("../data/cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

## 01. Padronização dos dados de texto

Precisamos de uma função que faça a limpeza dos campos de texto da base de dados com o objetivo de padronizar os valores. Por exemplo, 'Centro Cívico' e 'centro civico' devem ser tratados como iguais. Para isso, iremos remover a acentuação, converter tudo para letras minúsculas e eliminar caracteres especiais, mantendo apenas letras, números e espaços. 

In [5]:
def limpar_texto(valor):
    if pd.isna(valor):
        return pd.NA
    
    valor = str(valor).strip().lower()
    
    valor = unicodedata.normalize("NFKD", valor)
    valor = valor.encode("ascii", "ignore").decode("utf-8")
    
    valor = re.sub(r"[^a-z0-9\s]", " ", valor)
    valor = re.sub(r"\s+", " ", valor).strip()
    
    return valor if valor else pd.NA


Essa função irá executar:

1. Tratamento de valores nlos: se o valor for NaN, a função retorna `pd.NA`
2. Conversão para string e limpeza básica: o valor é convertido para string, removeremos espaços extras no início e no fim (`.strip()`) e transformamos tudo em letras minúsculas (.lower())
3. Remoção de acentos: usamos `unicodedata.normalize("NFKD", valor)` para decompor os caracteres acentuados (ex: "ção" vira "c~a~o"), e depois `.encode("ascii", "ignore")` elimina os acentos, mantendo apenas os caracteres ASCII básicos.
4. Filtragem de caracteres: com `re.sub(r"[^a-z0-9\s]", " ", valor)`, substituímos qualquer caractere que não seja letra minúscula, número ou espaço por um espaço em branco. Isso remove pontuação, símbolos e outros caracteres especiais. Esse espaço em branco é normalizado com `re.sub(r"\s+", " ", valor).strip()`, onde é substituida sequência de espaços por um único espaço e `.strip()` remove os espaços desnecessários das bordas. 

Ao fim, se toda a limpeza retornar uma string vazia, retornamos pd.NA. Caso contrário, retornamos o texto limpo.

## 02. Padronização de colunas binárias

Para padronizar colunas binárias, definindo `0` como `False` e `1` como `True`, é necessário primeiro identificar todos os valores distintos presentes na base de dados. Isso garante que o mapeamento seja feito corretamente. Para identificar todos os valores únicos na tupla `COLUNAS_BINARIAS`, disponibilizado em `config/data_cleaning_config.py`, faremos:

In [11]:
valores_unicos = {col: set() for col in COLUNAS_BINARIAS}

In [14]:
for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)
    
    for col in COLUNAS_BINARIAS:
        if col in df.columns:
            valores = df[col].dropna().unique()
            valores_unicos[col].update(valores)

for col, valores in valores_unicos.items():
    print(f"\n{col}:")
    print(sorted(valores))


FLAG_EQUIPAMENTO_URBANO:
['-----------------------', 'N', 'NÃO', 'SIM', 'Y', 'f', 't']

FLAG_FLAGRANTE:
['--------------', 'NÃO', 'NÃ\x83O', 'SIM']

NATUREZA1_DEFESA_CIVIL:
['----------------------', '0', '0.0', '1', '1.0']

NATUREZA2_DEFESA_CIVIL:
['----------------------', '0', '0.0', '1', '1.0']

NATUREZA3_DEFESA_CIVIL:
['----------------------', '0', '0.0', '1']

NATUREZA4_DEFESA_CIVIL:
['----------------------', '0', '0.0', '1']

NATUREZA5_DEFESA_CIVIL:
['----------------------', '0', '0.0']


In [ ]:
def mapear_binario(valor):

    if pd.isna(valor):
        return 0
    
    s = str(valor).strip().lower()

    if s in ('1.0', '0.0'):
        return 1 if s == '1.0' else 0

    valor_limpo = limpar_texto(valor)

    if valor_limpo in ('sim', 'y', 't', '1'):
        return  1
    
    if valor_limpo in ('n', 'nao', 'f', '0', '-----------------------',
                        '--------------', '----------------------',
                        '----------------------', '----------------------', 
                        '----------------------', '----------------------'):
        return 0
    
    return 0

## 01. Analisando colunas faltantes

Vamos verificar se todas as fontes de dados possuem as mesmas colunas. Para isso, utilizaremos o arquivo `2017-01-01_sigesguarda_-_Base_de_Dados.csv` como referência e extrairemos os cabeçalhos dos demais arquivos CSV, a fim de identificar possíveis colunas faltantes.

In [6]:
def ler_cabecalho(caminho):
    df = pd.read_csv(caminho, nrows=0, encoding="utf-8", sep=";")
    return list(df.columns)

In [7]:
cabecalhos = {arq: ler_cabecalho(arq) for arq in arquivos}

In [8]:
nome_referencia = "2017-01-01_sigesguarda_-_Base_de_Dados.csv"
caminho_referencia = data_dir / nome_referencia

In [9]:
ref_colunas = cabecalhos[caminho_referencia]

In [10]:
for arquivo, colunas in cabecalhos.items():
    if arquivo == caminho_referencia:
        continue 
    if colunas != ref_colunas:
        print(f"Diferença encontrada em: {arquivo.name}")
        print("Colunas faltando:", set(ref_colunas) - set(colunas))
        print("Colunas extras:", set(colunas) - set(ref_colunas))
        print("-" * 50)

Diferença encontrada em: 2024-04-24_sigesguarda_-_Base_de_Dados.csv
Colunas faltando: set()
Colunas extras: set()
--------------------------------------------------
Diferença encontrada em: 2024-07-01_sigesguarda_-_Base_de_Dados.csv
Colunas faltando: {'ATENDIMENTO_ANO'}
Colunas extras: set()
--------------------------------------------------
Diferença encontrada em: 2024-12-01_sigesguarda_-_Base_de_Dados.csv
Colunas faltando: {'ATENDIMENTO_ANO'}
Colunas extras: set()
--------------------------------------------------
Diferença encontrada em: 2024-07-09_sigesguarda_-_Base_de_Dados.csv
Colunas faltando: {'ATENDIMENTO_ANO'}
Colunas extras: set()
--------------------------------------------------
Diferença encontrada em: 2024-09-01_sigesguarda_-_Base_de_Dados.csv
Colunas faltando: {'ATENDIMENTO_ANO'}
Colunas extras: set()
--------------------------------------------------


Os arquivos a seguir não possuem a coluna 'ATENDIMENTO_ANO':

- `2024-04-24_sigesguarda_-_Base_de_Dados`; 
- `2024-07-01_sigesguarda_-_Base_de_Dados`;
- `2024-12-01_sigesguarda_-_Base_de_Dados`; 
- `2024-07-09_sigesguarda_-_Base_de_Dados`; 
- `2024-09-01_sigesguarda_-_Base_de_Dados`;

Para correção, considerando que o ano de atendimento corresponde ao ano de disponibilização da base, propõe-se criar a coluna 'ATENDIMENTO_ANO' e preenchê-la com o valor 2024 para todos os registros desses arquivos.